# 2.2. Chuẩn hóa văn bản, ký hiệu toán học và khử trùng xuyên phiên bản

## 1. Quy tắc làm sạch LaTeX và đồng nhất biểu thức toán (`normalize.py`)

- **Gỡ lệnh trình bày và giữ nguyên ngữ nghĩa**: Bóc đệ quy các lệnh định dạng chữ (`\textbf`, `\emph`, `\caption`, `\footnote`...) để lấy phần nội dung văn bản; đổi `\href{url}{text}` thành `text (url)`; loại bỏ các lệnh căn chỉnh bố cục thuần túy (`\centering`, `\toprule`, `\midrule`, `\bottomrule`, `\vspace`, `\label`...) và chuẩn hóa bảng mã Unicode về dạng `NFC`.
- **Đồng nhất ký hiệu toán học**: Giữ nguyên khoảng trắng bên trong `\text{...}` và `\mbox{...}`, rút gọn khoảng trắng thừa quanh toán tử để tăng tỷ lệ khớp mã băm giữa các phiên bản, đưa toàn bộ công thức nội dòng (`$...$`, `\(...\)`) về `$...$` và đưa toàn bộ công thức khối (`$$...$$`, `\[...\]`, `equation*`, `align`, `gather`, `multline`...) về `\begin{equation}...\end{equation}`.

## 2. Khử trùng hai tầng xuyên phiên bản (`bibliography.py` và `deduplicate.py`)

- **Tầng 1 - Hợp nhất mục tham khảo và đồng bộ khóa `\cite{}`**: Ghép nối các mục từ `.bib`, `\bibitem` và tệp `.bbl` dựa trên định danh (`doi`, `eprint` arXiv) hoặc cặp khóa chuẩn hóa `(title, authors)`, hợp nhất đầy đủ các trường thông tin từ mọi phiên bản và cập nhật lại toàn bộ lệnh `\cite{}` (bao gồm cả lệnh nằm trong công thức toán) về một khóa đại diện duy nhất.
- **Tầng 2 - Kho phần tử `elements` dùng chung theo mã băm SHA-256**: Sau khi chuẩn hóa văn bản và đồng bộ khóa trích dẫn, mỗi phần tử được tính mã băm theo bộ ba `(arxiv_id, kind, normalized_content)` để lưu đúng một bản duy nhất trong bảng `elements`, trong khi cấu trúc cây của từng phiên bản được biểu diễn bằng các cạnh `(version, element_id, parent_id)` trong bảng `hierarchy`.

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
from parser.normalize import normalize
from parser.bibliography import parse_bibitems, parse_bibtex, unify_entries
from parser.hierarchy import parse_hierarchy
from parser.deduplicate import deduplicate

norm_samples = [
    ("Lệnh định dạng chữ, \\href và \\cite", r"We follow \textbf{Vaswani et al.} \href{https://arxiv.org/abs/1706.03762}{\emph{Transformer}} \cite{vit_transformer}.", "sentence"),
    ("Bảng biểu và xóa lệnh kẻ khung trang trí", r"\centering\begin{tabular}{cc}\toprule A & B \\ \midrule 1 & 2 \\ \bottomrule\end{tabular}", "figure"),
    ("Đồng nhất công thức nội dòng và toán tử", r"Given \( a + b = c \) and $x_1 + x_2 = y$.", "sentence"),
    ("Đồng nhất công thức khối và bảo vệ \\text{}", r"E = m c^2 + p c \quad \text{in Eq.} ~ \cite{vit_transformer}", "block_formula"),
]
df_norm = pd.DataFrame([
    {
        "Trường hợp minh họa": name,
        "Đầu vào LaTeX gốc": raw,
        "Đầu ra sau chuẩn hóa và đổi khóa": normalize(raw, {"vit_transformer": "vaswani2017"}, kind=k),
    }
    for name, raw, k in norm_samples
])
display(style_table(df_norm, caption="Bảng 7.1. Đối chiếu đầu vào LaTeX gốc và kết quả sau chuẩn hóa bằng hàm normalize()."))

v1_tex = r"""\documentclass{article}\title{Citation Demo v1}\begin{document}
\section{Introduction}
We study \(a + b = c\) following \textbf{Vaswani et al.} \cite{vaswani2017}.
\begin{thebibliography}{9}
\bibitem{vaswani2017} A. Vaswani and N. Shazeer.\newblock Attention Is All You Need.\newblock 2017.
\end{thebibliography}\end{document}"""

v2_tex = r"""\documentclass{article}\title{Citation Demo v2}\begin{document}
\section{Background and Model}
We study $a+b=c$ following {Vaswani et al.} \cite{vit_transformer}.
\begin{align*}E = m c^2 + p c \quad \text{in Eq.} ~ \cite{vit_transformer}\end{align*}
\end{document}"""
v2_bib = r"""@article{vit_transformer,
  author = {A. Vaswani and N. Shazeer},
  title = {Attention Is All You Need},
  year = {2017},
  doi = {10.5555/3295222.3295349}
}"""

v3_tex = r"""\documentclass{article}\title{Citation Demo v3}\begin{document}
\section{Background and Model}
We study $a + b = c$ following \emph{Vaswani et al.} \cite{vit_transformer}.
\[ E=m c^2+p c \quad \text{in Eq.} ~ \cite{vit_transformer} \]
\end{document}"""
v3_bib = r"""@inproceedings{vit_transformer,
  author = {Vaswani, A. and Shazeer, N.},
  title = {Attention Is All You Need},
  journal = {NeurIPS},
  url = {https://arxiv.org/abs/1706.03762}
}"""

h1, h2, h3 = parse_hierarchy(v1_tex), parse_hierarchy(v2_tex), parse_hierarchy(v3_tex)
raw_entries = [*parse_bibitems(v1_tex, 1), *parse_bibtex(v2_bib, 2), *parse_bibtex(v3_bib, 3)]
bib_rows, mappings, conflicts = unify_entries("2310.12345", raw_entries)
els, edges, _ = deduplicate("2310.12345", {1: h1, 2: h2, 3: h3}, mappings)

display(style_table(
    pd.DataFrame(bib_rows).drop(columns=["arxiv_id"]),
    caption="Bảng 7.2. Bản ghi bibtex duy nhất sau khi hợp nhất đầy đủ trường thông tin từ ba phiên bản.",
    col_formats={"year": lambda v: "-" if pd.isna(v) else str(int(v))},
))

short_id = {e["element_id"]: e["element_id"][:21] + "..." for e in els}
display(style_table(
    pd.DataFrame([{"element_id (rút gọn)": short_id[e["element_id"]], "kind": e["kind"], "content": e["content"]} for e in els]),
    caption=f"Bảng 7.3. Kho phần tử dùng chung xuyên ba phiên bản ({len(h1.nodes)+len(h2.nodes)+len(h3.nodes)} nút ban đầu giảm còn {len(els)} phần tử duy nhất).",
))

kind_by_id = {e["element_id"]: e["kind"] for e in els}
display(style_table(
    pd.DataFrame([
        {
            "version": ed["version"],
            "child_kind": kind_by_id[ed["element_id"]],
            "element_id": short_id[ed["element_id"]],
            "parent_kind": kind_by_id[ed["parent_id"]],
            "parent_id": short_id[ed["parent_id"]],
        }
        for ed in edges
    ]),
    caption="Bảng 7.4. Bảng cạnh hierarchy theo từng phiên bản: cùng một câu văn gắn vào mục Introduction ở v1 và gắn vào Background and Model ở v2, v3.",
    int_cols=["version"],
))

## 3. So sánh tỷ lệ khử trùng giữa nhóm bài đơn phiên bản và nhóm bài đa phiên bản

In [ ]:
rg = m2["dedup_by_revision_group"]
df_rg = pd.DataFrame([
    {
        "Nhóm bài báo": "Bài chỉ có 1 phiên bản (v1 duy nhất)",
        "Số bài báo": rg["single_version"]["publications"],
        "Elements trước": rg["single_version"]["elements_before"],
        "Elements sau": rg["single_version"]["elements_after"],
        "Giảm Elements (%)": 100.0 * rg["single_version"]["elements_reduction_rate"],
        "BibTeX trước": rg["single_version"]["bibtex_before"],
        "BibTeX sau": rg["single_version"]["bibtex_after"],
        "Giảm BibTeX (%)": 100.0 * rg["single_version"]["bibtex_reduction_rate"],
    },
    {
        "Nhóm bài báo": "Bài có từ 2 phiên bản trở lên (v1 đến vN)",
        "Số bài báo": rg["multi_version"]["publications"],
        "Elements trước": rg["multi_version"]["elements_before"],
        "Elements sau": rg["multi_version"]["elements_after"],
        "Giảm Elements (%)": 100.0 * rg["multi_version"]["elements_reduction_rate"],
        "BibTeX trước": rg["multi_version"]["bibtex_before"],
        "BibTeX sau": rg["multi_version"]["bibtex_after"],
        "Giảm BibTeX (%)": 100.0 * rg["multi_version"]["bibtex_reduction_rate"],
    },
    {
        "Nhóm bài báo": "Toàn tập hợp nhất (25.000 bài báo)",
        "Số bài báo": scope["papers"],
        "Elements trước": m2["deduplication"]["elements"]["before"],
        "Elements sau": m2["deduplication"]["elements"]["after"],
        "Giảm Elements (%)": 100.0 * m2["deduplication"]["elements"]["reduction_rate"],
        "BibTeX trước": m2["deduplication"]["bibtex"]["before"],
        "BibTeX sau": m2["deduplication"]["bibtex"]["after"],
        "Giảm BibTeX (%)": 100.0 * m2["deduplication"]["bibtex"]["reduction_rate"],
    },
])
display(style_table(
    df_rg,
    caption="Bảng 7.5. Tỷ lệ khử trùng phân tách theo số phiên bản: trên 10.933 bài đa phiên bản, số dòng giảm 46,95% ở elements và 50,41% ở bibtex.",
    int_cols=["Số bài báo", "Elements trước", "Elements sau", "BibTeX trước", "BibTeX sau"],
    pct_cols=["Giảm Elements (%)", "Giảm BibTeX (%)"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
xr = np.arange(3)
wr = 0.34
ax1.bar(xr - wr / 2, df_rg["Giảm Elements (%)"].to_numpy(dtype=float), wr, color=BOOK_PALETTE["navy"], label="Giảm Elements (%)")
ax1.bar(xr + wr / 2, df_rg["Giảm BibTeX (%)"].to_numpy(dtype=float), wr, color=BOOK_PALETTE["steel"], label="Giảm BibTeX (%)")
ax1.set_xticks(xr)
ax1.set_xticklabels(["1 phiên bản\n(14.067 bài)", "Từ 2 phiên bản\n(10.933 bài)", "Toàn tập\n(25.000 bài)"])
ax1.set_ylim(0, 60)
ax1.grid(axis="y")
ax1.legend(fontsize=8.1)
ax1.set_ylabel("Tỷ lệ bản sao loại bỏ (%)")
ax1.set_title("Hình 7.1a. Hiệu quả khử trùng theo nhóm bài báo")
for idx_g, r_g in df_rg.iterrows():
    ax1.text(idx_g - wr / 2, float(r_g["Giảm Elements (%)"]) + 1.1, fmt_pct(r_g["Giảm Elements (%)"], 1), ha="center", fontsize=7.9)
    ax1.text(idx_g + wr / 2, float(r_g["Giảm BibTeX (%)"]) + 1.1, fmt_pct(r_g["Giảm BibTeX (%)"], 1), ha="center", fontsize=7.9)

if "bibtex_profile" in m2:
    bp = m2["bibtex_profile"]["field_nonnull_rates"]
    s_bp = pd.Series({str(k): float(v) * 100.0 for k, v in bp.items()}).sort_values(ascending=True)
    yb = np.arange(len(s_bp))
    ax2.barh(yb, s_bp.to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.52)
    ax2.set_yticks(yb)
    ax2.set_yticklabels([str(k) for k in s_bp.index])
    ax2.set_xlim(0, 115)
    ax2.grid(axis="x")
    ax2.set_xlabel("Tỷ lệ có dữ liệu (%)")
    ax2.set_title("Hình 7.1b. Độ phủ trường trên 1.177.293 mục BibTeX")
    for i_b, v_b in enumerate(s_bp.values):
        ax2.text(float(v_b) + 1.5, i_b, fmt_pct(v_b, 1), va="center", fontsize=8.0)
plt.tight_layout()
plt.show()

```{note}
**Ghi nhận các trường hợp biên trong quá trình hợp nhất:**
Trên toàn tập 25.000 bài báo có 1.206 phiên bản (1.026 phiên bản lỗi cú pháp và 180 phiên bản thiếu nguồn từ M1) được tách riêng vào bộ đếm `versions_with_unmeasured_counts` để không làm sai lệch mẫu số trước và sau khử trùng. Hệ thống cũng ghi nhận 75.595 khóa trích dẫn không có mục tương ứng trong tệp tài liệu tham khảo của tác giả và 69.929 trường hợp khác biệt giá trị trường giữa các phiên bản (ưu tiên giữ giá trị xuất hiện ở phiên bản sớm nhất kèm nhật ký đối soát).
```

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/normalize.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/normalize.py).
- Mã nguồn triển khai: [`src/parser/bibliography.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/bibliography.py).
- Mã nguồn triển khai: [`src/parser/deduplicate.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/deduplicate.py).
- Mã nguồn triển khai: [`src/parser/statistics.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/statistics.py).